In [1]:
import pandas as pd
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory

In [2]:
produtos = ['A','B']
maquinas = ['m1','m2']

maximos_maquinas={
    'm1':24,
    'm2':16
}

informações = {
    'A':{
        'm1':4,
        'm2':4,
        'lucro':80,
    },
    'B':{
        'm1':6,
        'm2':2,
        'lucro':60,
    }
}

In [3]:
model = pyo.ConcreteModel()

model.produtos = pyo.Set(initialize=produtos)
model.maquinas = pyo.Set(initialize=maquinas)
model.x = pyo.Var(model.produtos, bounds=(0,None), domain=pyo.NonNegativeIntegers)

def objetivo(model):
    return sum(
        model.x[p] *informações[p]['lucro'] for p in model.produtos
    )
model.obj = pyo.Objective(rule=objetivo, sense=pyo.maximize)

def restricao_maquinas(model, m):
    return sum(
        model.x[p]* informações[p][m]  for p in model.produtos
    ) <= maximos_maquinas[m]
model.restricao_maquinas = pyo.Constraint(model.maquinas,rule = restricao_maquinas)

def restricao_demanda_b(model):
    return model.x['B'] <= 3
model.restricao_demanda_b = pyo.Constraint(rule = restricao_demanda_b)


In [4]:
model.pprint()

2 Set Declarations
    maquinas : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {'m1', 'm2'}
    produtos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {'A', 'B'}

1 Var Declarations
    x : Size=2, Index=produtos
        Key : Lower : Value : Upper : Fixed : Stale : Domain
          A :     0 :  None :  None : False :  True : NonNegativeIntegers
          B :     0 :  None :  None : False :  True : NonNegativeIntegers

1 Objective Declarations
    obj : Size=1, Index=None, Active=True
        Key  : Active : Sense    : Expression
        None :   True : maximize : 80*x[A] + 60*x[B]

2 Constraint Declarations
    restricao_demanda_b : Size=1, Index=None, Active=True
        Key  : Lower : Body : Upper : Active
        None :  -Inf : x[B] :   3.0 :   True
    restricao_maquinas : Size=2, Index=maquinas, Active=True
        Key : Lo

In [5]:
opt = SolverFactory('cplex')
resultado = opt.solve(model, tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\DECIV\AppData\Local\Temp\tmp41y1y8rg.cplex.log' open.
CPLEX> Problem 'C:\Users\DECIV\AppData\Local\Temp\tmpn12w792i.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\DECIV\AppData\Local\Temp\tmpn12w792i.pyomo.lp
Objective sense      : Maximize
Variables            :       2  [General Integer: 2]
Objective nonzeros   :       2
Linear constraints   :       3  [Less: 3]
  Nonzeros           :       5
  RHS nonzeros       :       3

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 60.00000       

In [6]:
print(model.x.pprint())
print(pyo.value(model.obj))

x : Size=2, Index=produtos
    Key : Lower : Value : Upper : Fixed : Stale : Domain
      A :     0 :   3.0 :  None : False : False : NonNegativeIntegers
      B :     0 :   2.0 :  None : False : False : NonNegativeIntegers
None
360.0


In [8]:
for m in model.x:
    print(f"Quantidade de {m}: {pyo.value(model.x[m])}")

Quantidade de A: 3.0
Quantidade de B: 2.0
